# GSM8K dataset preparation

Load the official GSM8K splits, normalize their records, and create a reproducible development subset from training data. No GPU is required.

In [ ]:
from pathlib import Path
import os

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    !git clone {REPOSITORY_URL} {REPOSITORY_DIR}
os.chdir(REPOSITORY_DIR)
%pip install -q -r requirements.txt

In [ ]:
from src.data.gsm8k import load_gsm8k

DEVELOPMENT_SIZE = 100
RANDOM_SEED = 42
gsm8k = load_gsm8k(development_size=DEVELOPMENT_SIZE, seed=RANDOM_SEED)
print({name: len(split) for name, split in gsm8k.items()})

In [ ]:
EXPECTED_FIELDS = {"id", "question", "reference_solution", "reference_answer"}
assert set(gsm8k["train"].column_names) == EXPECTED_FIELDS
assert set(gsm8k["test"].column_names) == EXPECTED_FIELDS
assert len(gsm8k["development"]) == DEVELOPMENT_SIZE
assert all(row["id"].startswith("gsm8k-train-") for row in gsm8k["development"])
assert all(row["id"].startswith("gsm8k-test-") for row in gsm8k["test"].select(range(5)))
print("Split isolation and normalized schema validated.")

In [ ]:
gsm8k["development"].select(range(3)).to_pandas()

In [ ]:
repeat = load_gsm8k(development_size=DEVELOPMENT_SIZE, seed=RANDOM_SEED)
assert repeat["development"]["id"] == gsm8k["development"]["id"]
print(f"Deterministic sampling confirmed with seed {RANDOM_SEED}.")

## Persist the normalized data and provenance

The next cell stores all normalized train, test, and development records in Hugging Face's on-disk Arrow format. It also records fingerprints, selected development IDs, schema, seed, package version, and repository commit in a JSON manifest. Google Drive is enabled by default because `/content` is temporary.

In [ ]:
from datetime import datetime, timezone
from datasets import DatasetDict, __version__ as datasets_version
import json
import subprocess

PERSIST_TO_GOOGLE_DRIVE = True
if PERSIST_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/PSSN2/artifacts")
else:
    ARTIFACT_ROOT = REPOSITORY_DIR / "artifacts"

snapshot_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
snapshot_dir = ARTIFACT_ROOT / "datasets" / "gsm8k" / snapshot_id
data_dir = snapshot_dir / "normalized_dataset"
snapshot_dir.mkdir(parents=True, exist_ok=False)
DatasetDict(gsm8k).save_to_disk(str(data_dir))

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
manifest = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository_url": REPOSITORY_URL,
    "repository_commit": git_commit,
    "dataset_id": "openai/gsm8k",
    "dataset_config": "main",
    "source_revision": "main",
    "datasets_package_version": datasets_version,
    "development_source_split": "train",
    "development_size": DEVELOPMENT_SIZE,
    "random_seed": RANDOM_SEED,
    "schema_fields": sorted(EXPECTED_FIELDS),
    "split_sizes": {name: len(split) for name, split in gsm8k.items()},
    "split_fingerprints": {
        name: getattr(split, "_fingerprint", None) for name, split in gsm8k.items()
    },
    "development_example_ids": list(gsm8k["development"]["id"]),
    "storage_format": "huggingface_datasetdict_arrow",
    "data_directory": str(data_dir),
}
manifest_path = snapshot_dir / "gsm8k_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(f"Normalized dataset saved to: {data_dir}")
print(f"Dataset manifest saved to: {manifest_path}")